# Pendulum Initial Condition and Friction Identification
This notebook converts the Julia-based pendulum identification workflow into Python. We recover the unknown initial state and friction coefficient from noisy pendulum data.

In [1]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

np.random.seed(14588)

We consider the pendulum model:

\[ \frac{d}{dt} x(t) = \begin{bmatrix} x_2(t) \\ -\frac{g}{L}\sin(x_1(t)) - \frac{b}{m} x_2(t) \end{bmatrix} \]

and identify both the unknown friction coefficient $b$ and the unknown initial state $x_0$ from noisy measurements.

In [2]:
# Model parameters defined as constants
g = 9.81  # gravity constant
L = 1.0   # length of pendulum
b = 0.25  # true damping coefficient
m = 0.5   # mass of pendulum

x0_true = [0.0, np.pi / 2]  # true initial state
tspan = (0.0, 4.0)
t_eval = np.arange(tspan[0], tspan[1] + 1e-8, 0.05)

Simulate the true pendulum trajectory and add measurement noise to both state variables.

In [3]:
def simple_pendulum(t, x):
    theta, omega = x
    return [omega, -(g / L) * np.sin(theta) - (b / m) * omega]

sol = solve_ivp(simple_pendulum, tspan, x0_true, t_eval=t_eval, vectorized=False)

t = sol.t
x = sol.y
sigma = 0.05
y = x + sigma * np.random.randn(*x.shape)

Define the training model and the cost function that optimizes over the unknown initial state and friction.

In [ ]:
def simple_pendulum_train(t, x, friction):
    theta, omega = x
    return [omega, -(g / L) * np.sin(theta) - (friction / m) * omega]


def simulate_training(w):
    u0 = [w[0], w[1]] #two state variables are initial angle and initial angular velocity
    friction = float(w[2])
    sol = solve_ivp(lambda tt, xx: simple_pendulum_train(tt, xx, friction), tspan, u0, t_eval=t_eval, vectorized=False)
    return sol.y


def cost_function(w):
    x_pred = simulate_training(w)
    return np.sum((y - x_pred) ** 2)

Solve the optimization problem with a local optimizer, starting from a rough initial guess.

In [ ]:
initial_guess = [0.0, 0.0, 1.0] #states and friction
res = minimize(cost_function, initial_guess, method='BFGS', options={'disp': True})
res

Optimization terminated successfully.
         Current function value: 0.389017
         Iterations: 25
         Function evaluations: 165
         Gradient evaluations: 40


  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 0.3890168023324351
        x: [ 3.566e-03  1.559e+00  2.435e-01]
      nit: 25
      jac: [ 4.172e-07  1.013e-06  1.840e-06]
 hess_inv: [[ 2.932e-03  5.490e-04  3.920e-05]
            [ 5.490e-04  5.875e-02  1.690e-02]
            [ 3.920e-05  1.690e-02  8.318e-03]]
     nfev: 165
     njev: 40

In [ ]:
# Print the estimated initial state and friction parameter
x0_est = res.x[:2]
b_est = res.x[2]

print('Found initial conditions:', np.round(x0_est, 4))
print('Ground truth:', np.round(x0_true, 4))
print('Found friction parameter:', np.round(b_est, 4))
print('Ground truth:', b)

Visualize the fit between the identified model and the noisy ground truth data.

In [ ]:
sol_est = solve_ivp(lambda tt, xx: simple_pendulum_train(tt, xx, b_est), tspan, x0_est, t_eval=t_eval, vectorized=False)
x_est = sol_est.y

plt.figure(figsize=(8, 4))
plt.plot(t, x_est.T[:, 0], label=r'$\hat{x}_1$ (PEM)', lw=2)
plt.plot(t, x_est.T[:, 1], label=r'$\hat{x}_2$ (PEM)', lw=2)
plt.plot(t, y.T[:, 0], '--', label=r'$\theta$ (noisy)', alpha=0.8)
plt.plot(t, y.T[:, 1], '--', label=r'$\dot{\theta}$ (noisy)', alpha=0.8)
plt.xlabel('Time in s')
plt.ylabel('States')
plt.title('Identified model vs. ground truth')
plt.legend()
plt.grid(True)
plt.show()